In [19]:
# Install the dependencies
%pip install anthropic python-dotenv dotenv

Note: you may need to restart the kernel to use updated packages.


In [20]:
#Load the env variables
from dotenv import load_dotenv
load_dotenv()



True

In [21]:
#Create API client 
from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-0"

In [23]:
#Make a request 
message = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = [
        {
            "role": "user",
            "content": "What is Quantum physics? Answer in one sentence."
        }
    ] 
)

message

message.content[0].text

#Make a request 
message = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = [
        {
            "role": "user",
            "content": "Write another sentence."
        }
    ] 
)

#Just to see that with the second request the context is maintained with the model or not. It is not maintained.
message.content[0].text

'The morning sun filtered through the kitchen window, casting golden rectangles across the worn wooden table where steam rose gently from a cup of freshly brewed coffee.'

In [ ]:
# Print the message generated by AI
message

Message(id='msg_01H5GTrMsMhrdyWXnHAyYnkn', container=None, content=[TextBlock(citations=None, text="I'm doing well and happy to chat with you today!", type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=20, output_tokens=15, server_tool_use=None, service_tier='standard'))

In [ ]:
# Extract only the reply from the message
message.content[0].text

'The autumn leaves drifted slowly to the ground, painting the sidewalk in shades of gold and crimson.'

-> Now we create the multi-turn conversation, where we create the context-window for the model. So basically our first question "What is the Quantum Physics" and its answer and our next question "Write another sentence" will be appended in a list and given to our agent. So it understands the previous context. This is basically called building a context window.

In [ ]:
def add_user_message(messages, text):
    '''Helper function to add user message to the messages list'''
    user_message =  {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_messages(messages,text):
    '''Helper function to add assistant message to the messages list'''
    assistant_message =  {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    '''Helper function to make a request to the model and get the response. 
    Messages contains the past conversation i.e. qustions of both user and assistant asnwers.'''
    message = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    )

    return message.content[0].text

In [ ]:
#Make a starting list of the messages
messages = []

#Add a user message to the messages list
add_user_message(messages, "What is Quantum physics? Answer in one sentence.")

# messages

#Get the response from the model
answer = chat(messages)

# answer

#Add the assistant message to the messages list
add_assistant_messages(messages, answer)

# messages

#Add another user message to the messages list
add_user_message(messages, "Write another sentence.")

# messages

#Get the response from the model
answer = chat(messages)
answer

'At the quantum level, particles can be "entangled" with each other across vast distances, meaning that measuring one particle instantly affects its partner regardless of the space between them, defying our everyday understanding of how the world works.'

In [ ]:
#The whole multi-turn conversation in loop:

# Make an initial list of messages
messages = []

# Use a while loop to have a multi-turn conversation with the model and run the chatbot forever
while True:
    #Take the user input
    user_input = input("User: ")
    print(">", user_input)

    #Add the user message to the messages list
    add_user_message(messages, user_input)

    #Call the claude with the chat function
    answer = chat(messages)

    #Add generated text to the messages list
    add_assistant_messages(messages, answer)

    #Print the generated text
    print("Claude: ", answer)

> What is 2 + 2
Claude:  2 + 2 = 4
> Add 2 to the previous answer 
Claude:  4 + 2 = 6


Now we override the chat function. Here we give the system prompts to our Agent as well which is a guidence for the agent on how to product replies.

In [ ]:
def chat(messages, model = "claude-sonnet-4-0",  system = None, temperature = 0.7, stop_sequences = []):
    '''Helper function to make a request to the model and get the response. 
    Messages contains the past conversation i.e. qustions of both user and assistant asnwers.'''

   # Build dictionary first
    kwargs = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    # Only add system if not None
    if system is not None:
        kwargs["system"] = system

    # Now call the API once
    response = client.messages.create(**kwargs)

    return response.content[0].text

In [ ]:
messages = []

system = """You are a patient math tutor.
    You will help the user to solve math problems step by step."""
add_user_message(messages, "What is 2+2?")
answer = chat(messages, system = system, temperature = 1.0)
answer

"I'd be happy to help you solve 2 + 2!\n\nLet me walk you through this step by step:\n\n**Step 1:** We have two numbers to add together: 2 and 2\n\n**Step 2:** When we add, we're combining the quantities\n- Think of it as: if you have 2 items, and someone gives you 2 more items, how many do you have in total?\n\n**Step 3:** Count it out:\n- Start with 2: 🔴🔴\n- Add 2 more: 🔴🔴 + 🔴🔴\n- Count all together: 1, 2, 3, 4\n\n**Answer:** 2 + 2 = 4\n\nThis is one of the basic addition facts that's helpful to memorize since you'll use it frequently in math!"

Now we do the response streaming. It means that the data generated by the Agent will come in the chunks, rather than the whole reply generated by agent coming together. This saves time and imporves UX.

In [ ]:
messages = []

add_user_message(messages, "Write a one sentence description of a fake database")

stream = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    stream = True
)

for event in stream:
    print(event)


RawMessageStartEvent(message=Message(id='msg_01DAuMqdTLJLAwPDXYzueR24', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='User', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Pro', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(

Here the data came from claude in chunks, in the form of different events. What contains the real message is 'RawContentBlockDeltaEvent'

In [ ]:
messages = []

add_user_message(messages, "Whrite a one sentence description of a fake database")

with client.messages.stream(
    model = model,
    max_tokens = 1000,
    messages = messages,
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()



ParsedMessage(id='msg_01MhbA4jSvbmSBbQmweHPB6H', container=None, content=[ParsedTextBlock(citations=None, text='"EmployeeDB" is a simulated corporate database containing fabricated employee records with randomly generated names, departments, salaries, and performance metrics used for testing and demonstration purposes.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=40, server_tool_use=None, service_tier='standard'))

MESSAGE PREFILLING STARTS FROM HERE
- Think of Message Prefilling as putting words directly into the AI’s mouth before it even starts "thinking."
- It is used when we should force our agent to take the strong stance on one side when asked a question about preference.

In [ ]:
messages = []

add_user_message(messages, "Is tea better or coffee?")

# We want our agent to take the stance that coffee is better so message prefilling
add_assistant_messages(messages, "Coffee is better because")

answer = chat(messages)

answer

' it has more caffeine and tastes great. But I think most people would say that coffee and tea both have their place, and which one is "better" really depends on your personal preferences, the situation, and what you\'re looking for.\n\nHere are some ways they differ:\n\n**Coffee tends to be better for:**\n- Higher caffeine content (if you need a stronger energy boost)\n- Rich, bold flavors\n- Social coffeehouse culture\n- Pairing with breakfast or desserts\n\n**Tea tends to be better for:**\n- More variety in flavors and caffeine levels\n- Potential health benefits from antioxidants\n- Gentler, more sustained energy\n- Relaxation and mindfulness rituals\n- Lower acidity\n\nWhat matters most to you when choosing a drink? That might help determine which one suits you better.'

STOP SEQUENCES:
- fores calude to end the response as soon as it generates a string matching yours top sequence.

In [ ]:
messages = []

add_user_message(messages, "Count from 1 to 10.")
answer = chat(messages, stop_sequences=["5"]) #Provide 5 as a stop sequence so that the model stops generating after 5.

answer

'1, 2, 3, 4, '

SYSTEM PROMPTS: Decides the role of the agents.
Here we make a prompt evaluation pipeline which suggests how to make the changes in the prompt, so it produces better results.
Steps:
Step1 : Build an initial system prompt 
Step2: Build a dataset of the questions similar to the questions that agent will receive in production.
Step3: Feed the questions to the agent and get the answers.
Step4: Pass all the answers through a grader and give each answer a score out of 10.
Step5: Take the mean of all scores and generate a total score out of 10
Step6: Make the changes in the prompt accordingly, after reviewing specific answers which got low score out of 10

This is our goal:
![alt text](a-1.png)

-> Create a prompt evaluation dataset with the help of claude agent. Using Haiku for faster and smaller tasks like this is appropriate.

In [25]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_messages(messages, "```json")
    text = chat(messages, model="claude-haiku-4-5", stop_sequences=["```"])
    return json.loads(text)

Get the result and dump it inside the dataset.json file made in the same directory as this file.

In [26]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

Now we have a test-dataset of user quetions. we also make the prompt now and then feed both the prompt and each question to the claude and see the output.

In [32]:
def run_prompt(test_case):
    """Merges the prompt with the test case input, then returns the result"""
    prompt = f"""
        Pleases solve the following task:
        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [36]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_messages(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [37]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result."""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return{
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [41]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_tes_case ewith each case"""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print (f"Average Score: {average_score}")
    return results


In [42]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 6.666666666666667


In [40]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Looking at this task, I need to understand the structure of S3 bucket ARNs and how regions are represented in them.\n\nS3 bucket ARNs have different formats:\n- Standard format: `arn:aws:s3:::bucket-name` (no region specified)\n- Object ARN format: `arn:aws:s3:::bucket-name/object-key` (no region specified)\n- Access Point ARN: `arn:aws:s3:region:account-id:accesspoint/access-point-name`\n- Multi-Region Access Point ARN: `arn:aws:s3::account-id:accesspoint/access-point-name`\n\nFor regular S3 bucket ARNs, the region is typically not included in the ARN itself. However, some S3 features like Access Points do include the region.\n\nHere's my solution:\n\n```python\ndef extract_s3_region(arn):\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN.\n    \n    Args:\n        arn (str): The S3 bucket ARN\n        \n    Returns:\n        str: The AWS region or 'us-east-1' if region cannot be determined\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n    